# Knowledge Graph Construction
**Capstone — Intelligent Legal Decision Support through Neuro-Symbolic AI**

This notebook builds the full Neo4j Knowledge Graph from your cleaned Fact Layer outputs.

### Before you start
1. Go to **https://neo4j.com/cloud/aura** → Start Free → Create Free Instance
2. Copy the **URI** and **Password** it gives you (shown only once)
3. Make sure these files are in your Google Drive root:
   - `clean_fact_layer.py`
   - `kg_preprocess_utils.py`
   - `populate_kg.py`
   - `compute_similar_facts.py`
   - `MERGED_fact_layer_final/` folder

### Stages
| Cell | What it does | Time |
|------|-------------|------|
| 1–3  | Install + Mount Drive + Connect | 2 min |
| 4    | Clean all JSONs | ~20 min |
| 5    | Load nodes + relationships into Neo4j | ~30 min |
| 6    | Compute SIMILAR_FACTS edges | ~2–3 hrs CPU / ~15 min GPU |
| 7    | Verify the graph | 1 min |

---
## Cell 1 — Install dependencies

In [2]:
!pip install -q neo4j sentence-transformers faiss-cpu tqdm
print("Done.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.5/331.5 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 91.4 MB/s eta 0:00:00
Done.


---
## Cell 2 — Mount Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

# Verify your scripts are there
import os
scripts = [
    'clean_fact_layer.py',
    'kg_preprocess_utils.py',
    'populate_kg.py',
    'compute_similar_facts.py',
]
for s in scripts:
    path = f'/content/drive/MyDrive/{s}'
    status = '✅' if os.path.exists(path) else '❌ MISSING'
    print(f'{status}  {s}')

merged = '/content/drive/MyDrive/MERGED_fact_layer_final'
if os.path.exists(merged):
    n = len([f for f in os.listdir(merged) if f.endswith('.json')])
    print(f'✅  MERGED_fact_layer_final/ ({n} JSON files)')
else:
    print('❌  MERGED_fact_layer_final/ — NOT FOUND')

Mounted at /content/drive
✅  clean_fact_layer.py
✅  kg_preprocess_utils.py
✅  populate_kg.py
✅  compute_similar_facts.py
✅  MERGED_fact_layer_final/ (26688 JSON files)


---
## Cell 3 — Your Aura credentials + connection test

⚠️ **Fill in your URI and Password below before running.**

In [4]:
# ── FILL THESE IN ─────────────────────────────────────────────────────────────
NEO4J_URI      = "neo4j+s://6a5432ba.databases.neo4j.io"   # from Aura dashboard
#NEO4J_USER     = "neo4j"
NEO4J_USER     = "6a5432ba"
NEO4J_PASSWORD = "AyGQQOdAig7Qhjw9Z_lpmonX-ZGiFviF1q8MKUMEwiQ"                       # from Aura dashboard
# ─────────────────────────────────────────────────────────────────────────────

from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
driver.close()
print("✅ Connected to Neo4j Aura successfully!")

✅ Connected to Neo4j Aura successfully!


---
## Cell 4 — Clean the Fact Layer JSONs

Reads every JSON from `MERGED_fact_layer_final`, applies all 7 fixes
(judges, statutes, citations, concepts, parties, arguments, temporal_year)
and writes cleaned versions to `CLEANED_fact_layer/`.

**Resumable** — if Colab dies, just re-run this cell. Already-cleaned files are skipped.

In [5]:
INPUT_DIR  = '/content/drive/MyDrive/MERGED_fact_layer_final'
OUTPUT_DIR = '/content/drive/MyDrive/CLEANED_fact_layer'

# Copy utils to working dir so clean_fact_layer.py can import it
!cp /content/drive/MyDrive/kg_preprocess_utils.py /content/kg_preprocess_utils.py
!cp /content/drive/MyDrive/clean_fact_layer.py    /content/clean_fact_layer.py

!python /content/clean_fact_layer.py "{INPUT_DIR}" "{OUTPUT_DIR}"

Total files   : 26688
Already done  : 26688
To process    : 0

Done. 26688 total cleaned cases in /content/drive/MyDrive/CLEANED_fact_layer


In [6]:
# Quick check — how many cleaned files were written?
cleaned = [f for f in os.listdir(OUTPUT_DIR) if f.endswith('.json')]
print(f'Cleaned files: {len(cleaned)}')

# Peek at one to make sure structure looks right
import json, random
sample_file = random.choice(cleaned)
with open(os.path.join(OUTPUT_DIR, sample_file)) as f:
    sample = json.load(f)

print(f'\nSample: {sample_file}')
print(f'  case_id       : {sample.get("case_id")}')
print(f'  temporal_year : {sample.get("temporal_year")}')
print(f'  quality_flag  : {sample.get("quality_flag")}')
print(f'  judges_cleaned: {sample.get("metadata", {}).get("judges_cleaned")}')
print(f'  statutes      : {len(sample.get("reasoning", {}).get("statutes_cleaned", []))} entries')
print(f'  concepts      : {sample.get("legal_concepts_cleaned")}')
print(f'  citations     : {[(c["citation_type"], c["citation"]) for c in sample.get("citations_cleaned", [])][:3]}')
print(f'  arguments     : {len(sample.get("arguments_with_ids", []))} nodes')
print(f'  parties       : {sample.get("parties_cleaned")}')

Cleaned files: 26689

Sample: 2016_71c8adba393b.json
  case_id       : 2016_71c8adba393b
  temporal_year : 2016
  quality_flag  : standard
  judges_cleaned: [{'name': 'S. Ispat', 'confidence': 'high'}, {'name': 'R.F. Nariman', 'confidence': 'high'}, {'name': 'A.K. Sikri', 'confidence': 'high'}]
  statutes      : 3 entries
  concepts      : []
  citations     : [('reporter', '2016 (12) SCC 719'), ('case_number', 'CIVIL APPEAL No. 2562 OF 2008'), ('case_number', 'CIVIL APPEAL No.8557 OF 2015')]
  arguments     : 3 nodes
  parties       : {'petitioner': {'name': 'C C E Raigad', 'name_normalized': 'c c e raigad', 'outcome_for_party': 'favorable'}, 'respondent': {'name': 'M S Ispat Metallics Industries', 'name_normalized': 'm s ispat metallics industries', 'outcome_for_party': 'unfavorable'}}


---
## Cell 5 — Populate Neo4j (nodes + relationships + citations)

Runs 4 stages:
1. **Constraints & indexes** — uniqueness constraints + query indexes
2. **Nodes** — Case, Statute, Topic, Judge, Party, LegalConcept, Argument
3. **Relationships** — INVOLVES, RELATES_TO, DECIDED_BY, HAS_PARTY, MAKES_ARGUMENT, APPLIES_CONCEPT
4. **Citations** — CITES edges where resolvable

**Safe to re-run** — uses MERGE everywhere, so no duplicates.

In [7]:
from neo4j import GraphDatabase
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
with driver.session() as s:
    s.run("MATCH (n) DETACH DELETE n")
    print("Graph cleared ✅")
driver.close()

Graph cleared ✅


In [ ]:
!cp /content/drive/MyDrive/populate_kg.py /content/populate_kg.py

!python /content/populate_kg.py \
    --folder   "{OUTPUT_DIR}" \
    --uri      "{NEO4J_URI}" \
    --user     "{NEO4J_USER}" \
    --password "{NEO4J_PASSWORD}"

Connecting to Neo4j at neo4j+s://6a5432ba.databases.neo4j.io ...
  Connected.

Loading JSONs from /content/drive/MyDrive/CLEANED_fact_layer ...


---
## Cell 6 — SIMILAR_FACTS edges

Generates sentence embeddings (issue + facts text), finds pairs with
cosine similarity ≥ 0.7 using FAISS, and creates SIMILAR_FACTS edges.

**Saves progress to Drive** — embeddings.npy and similar_pairs.json are
saved so if Colab dies you don't recompute from scratch.

> 💡 If Colab gave you a **T4 GPU** (check Runtime → Change runtime type),
> remove the `--cpu` flag. Drops from ~2–3 hrs to ~15 min.

In [ ]:
# Check if GPU is available
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
has_gpu = result.returncode == 0
print('GPU available:', has_gpu)
if has_gpu:
    print(result.stdout.split('\n')[8])  # GPU name line
    print('\n→ Remove --cpu from the next cell!')
else:
    print('→ No GPU. Will use CPU (~2-3 hrs). Consider switching runtime to T4.')

In [ ]:
# Cell 6a — install faiss-gpu (you have T4)
!pip install -q faiss-gpu
# Cell 6b — run SIMILAR_FACTS (T4 GPU, ~15 min)
!cp /content/drive/MyDrive/compute_similar_facts.py /content/compute_similar_facts.py

FOLDER     = '/content/drive/MyDrive/CLEANED_fact_layer'
EMB_PATH   = '/content/drive/MyDrive/embeddings.npy'
PAIRS_PATH = '/content/drive/MyDrive/similar_pairs.json'

!python /content/compute_similar_facts.py \
    --folder     "$FOLDER" \
    --uri        "$NEO4J_URI" \
    --user       "$NEO4J_USER" \
    --password   "$NEO4J_PASSWORD" \
    --threshold  0.75 \
    --emb-path   "$EMB_PATH" \
    --pairs-path "$PAIRS_PATH"

---
## Cell 7 — Verify the graph

Run this to confirm everything populated correctly.

In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

with driver.session() as s:

    print('=== NODE COUNTS ===')
    result = s.run("""
        MATCH (n)
        RETURN labels(n)[0] AS label, COUNT(n) AS count
        ORDER BY count DESC
    """)
    for row in result:
        print(f'  {row["label"]:20}  {row["count"]:>8}')

    print('\n=== RELATIONSHIP COUNTS ===')
    result = s.run("""
        MATCH ()-[r]->()
        RETURN type(r) AS rel, COUNT(r) AS count
        ORDER BY count DESC
    """)
    for row in result:
        print(f'  {row["rel"]:25}  {row["count"]:>8}')

    print('\n=== OUTCOME DISTRIBUTION ===')
    result = s.run("""
        MATCH (c:Case)
        RETURN c.outcome AS outcome, COUNT(c) AS count
        ORDER BY count DESC
    """)
    for row in result:
        print(f'  {str(row["outcome"]):10}  {row["count"]:>8}')

    print('\n=== SAMPLE CASE SUBGRAPH ===')
    result = s.run("""
        MATCH (c:Case)
        WITH c LIMIT 1
        OPTIONAL MATCH (c)-[:INVOLVES]->(s:Statute)
        OPTIONAL MATCH (c)-[:DECIDED_BY]->(j:Judge)
        OPTIONAL MATCH (c)-[:APPLIES_CONCEPT]->(lc:LegalConcept)
        RETURN c.title AS title,
               c.year AS year,
               c.outcome AS outcome,
               collect(DISTINCT s.statute_id)[..3] AS statutes,
               collect(DISTINCT j.name) AS judges,
               collect(DISTINCT lc.name) AS concepts
    """)
    for row in result:
        print(f'  title   : {row["title"]}')
        print(f'  year    : {row["year"]}')
        print(f'  outcome : {row["outcome"]}')
        print(f'  statutes: {row["statutes"]}')
        print(f'  judges  : {row["judges"]}')
        print(f'  concepts: {row["concepts"]}')

driver.close()
print('\n✅ KG is ready for the Symbolic Reasoning Engine (Stage 4).')

---
## Cell 8 — (Optional) If Colab dies before Cell 6 finishes

Run this to restore from saved embeddings + pairs without recomputing.

In [ ]:
# Only run this if Cell 6 died mid-way and you already have
# embeddings.npy + similar_pairs.json saved to Drive

EMB_PATH   = '/content/drive/MyDrive/embeddings.npy'
PAIRS_PATH = '/content/drive/MyDrive/similar_pairs.json'

if not os.path.exists(EMB_PATH) or not os.path.exists(PAIRS_PATH):
    print('❌ Files not found in Drive. Run Cell 6 from scratch.')
else:
    print('✅ Found saved files. Loading into Neo4j...')
    !python /content/compute_similar_facts.py \
        --folder     "{OUTPUT_DIR}" \
        --uri        "{NEO4J_URI}" \
        --user       "{NEO4J_USER}" \
        --password   "{NEO4J_PASSWORD}" \
        --emb-path   "{EMB_PATH}" \
        --pairs-path "{PAIRS_PATH}" \
        --cpu

In [ ]:
from neo4j import GraphDatabase
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

with driver.session() as s:

    # 1. Coverage — what % of cases have each relationship
    print("=== COVERAGE ===")
    total = 26688
    checks = [
        ("Cases with statutes",   "MATCH (c:Case) WHERE (c)-[:INVOLVES]->() RETURN COUNT(DISTINCT c) AS n"),
        ("Cases with concepts",   "MATCH (c:Case) WHERE (c)-[:APPLIES_CONCEPT]->() RETURN COUNT(DISTINCT c) AS n"),
        ("Cases with parties",    "MATCH (c:Case) WHERE (c)-[:HAS_PARTY]->() RETURN COUNT(DISTINCT c) AS n"),
        ("Cases with judges",     "MATCH (c:Case) WHERE (c)-[:DECIDED_BY]->() RETURN COUNT(DISTINCT c) AS n"),
        ("Cases with arguments",  "MATCH (c:Case) WHERE (c)-[:MAKES_ARGUMENT]->() RETURN COUNT(DISTINCT c) AS n"),
        ("Cases with SIMILAR",    "MATCH (c:Case) WHERE (c)-[:SIMILAR_FACTS]-() RETURN COUNT(DISTINCT c) AS n"),
    ]
    for label, cql in checks:
        n = s.run(cql).single()["n"]
        print(f"  {label:25} {n:>6} / {total}  ({100*n/total:.1f}%)")

    # 2. Statute quality — how many are resolved vs bare
    print("\n=== STATUTE QUALITY ===")
    r = s.run("MATCH (s:Statute) RETURN s.act_resolved AS resolved, COUNT(s) AS count")
    for row in r:
        print(f"  act_resolved={row['resolved']}  count={row['count']}")

    # 3. Concept distribution — most common
    print("\n=== TOP 10 LEGAL CONCEPTS ===")
    r = s.run("""
        MATCH (c:Case)-[:APPLIES_CONCEPT]->(lc:LegalConcept)
        RETURN lc.name AS concept, COUNT(c) AS cases
        ORDER BY cases DESC LIMIT 10
    """)
    for row in r:
        print(f"  {row['concept']:35} {row['cases']:>6} cases")

    # 4. Outcome balance
    print("\n=== OUTCOME BALANCE ===")
    r = s.run("MATCH (c:Case) RETURN c.outcome AS o, COUNT(c) AS n ORDER BY n DESC")
    for row in r:
        print(f"  {str(row['o']):20} {row['n']:>6}  ({100*row['n']/total:.1f}%)")

    # 5. SIMILAR_FACTS quality — score distribution
    print("\n=== SIMILARITY SCORE DISTRIBUTION ===")
    r = s.run("""
        MATCH ()-[r:SIMILAR_FACTS]->()
        RETURN
          round(r.similarity_score * 10) / 10 AS bucket,
          COUNT(r) AS count
        ORDER BY bucket
    """)
    for row in r:
        print(f"  score ~{row['bucket']}  count={row['count']}")

    # 6. Ground truth spot check — known case
    print("\n=== SPOT CHECK: look up a real case ===")
    r = s.run("""
        MATCH (c:Case)
        WHERE c.title CONTAINS 'Kesavananda'
           OR c.title CONTAINS 'Maneka Gandhi'
           OR c.title CONTAINS 'Olga Tellis'
        RETURN c.title, c.year, c.outcome,
               [(c)-[:INVOLVES]->(s) | s.statute_id][..3] AS statutes,
               [(c)-[:APPLIES_CONCEPT]->(lc) | lc.name][..3] AS concepts
        LIMIT 3
    """)
    rows = r.data()
    if rows:
        for row in rows:
            print(f"  {row['c.title'][:60]}")
            print(f"    year={row['c.year']} outcome={row['c.outcome']}")
            print(f"    statutes={row['statutes']}")
            print(f"    concepts={row['concepts']}")
    else:
        print("  Famous cases not found (may be named differently in corpus)")

driver.close()

In [ ]:
import json
from neo4j import GraphDatabase
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

with driver.session() as s:

    # First find the real case_id
    r = s.run("""
        MATCH (c:Case)
        WHERE c.title CONTAINS 'Kesavananda'
        RETURN c.case_id, c.title, c.year
        LIMIT 1
    """)
    row = r.single()
    if row:
        case_id = row["c.case_id"]
        print(f"Found: {row['c.title']} — case_id: {case_id}\n")

        # Now pull full subgraph
        r2 = s.run("""
            MATCH (c:Case {case_id: $cid})
            OPTIONAL MATCH (c)-[:INVOLVES]->(s:Statute)
            OPTIONAL MATCH (c)-[:APPLIES_CONCEPT]->(lc:LegalConcept)
            OPTIONAL MATCH (c)-[:HAS_PARTY]->(p:Party)
            OPTIONAL MATCH (c)-[:MAKES_ARGUMENT]->(a:Argument)
            OPTIONAL MATCH (c)-[:SIMILAR_FACTS]-(sim:Case)
            RETURN
                c.title AS title,
                c.year AS year,
                c.outcome AS outcome,
                c.issue AS issue,
                collect(DISTINCT s.statute_id) AS statutes,
                collect(DISTINCT lc.name) AS concepts,
                collect(DISTINCT p.name) AS parties,
                collect(DISTINCT a.text)[..3] AS sample_args,
                collect(DISTINCT sim.title)[..5] AS similar_cases
        """, cid=case_id)

        data = r2.single().data()
        print(f"Title   : {data['title']}")
        print(f"Year    : {data['year']}")
        print(f"Outcome : {data['outcome']}")
        print(f"Issue   : {data['issue']}")
        print(f"Statutes: {data['statutes']}")
        print(f"Concepts: {data['concepts']}")
        print(f"Parties : {data['parties']}")
        print(f"\nSample arguments:")
        for a in data['sample_args']:
            print(f"  - {a[:100]}")
        print(f"\nSimilar cases:")
        for t in data['similar_cases']:
            print(f"  - {t}")

driver.close()